<a href="https://colab.research.google.com/github/Rahilralu/pytorch/blob/main/CNN_fasion_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
from torchvision.datasets import FashionMNIST
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml
import numpy as np
torch.cuda.is_available()

True

In [2]:
X, y = fetch_openml("Fashion-MNIST",version=1,return_X_y=True,as_frame=False)

In [3]:
x_train,x_test,y_train,y_test = train_test_split(X,y,test_size=0.2)
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [4]:
x_train_scaled

array([[-0.00868013, -0.02413091, -0.03615267, ..., -0.15799991,
        -0.09046773, -0.03252005],
       [-0.00868013, -0.02413091, -0.03615267, ..., -0.15799991,
        -0.09046773, -0.03252005],
       [-0.00868013, -0.02413091, -0.03615267, ...,  3.39726083,
        -0.09046773, -0.03252005],
       ...,
       [-0.00868013, -0.02413091, -0.03615267, ..., -0.15799991,
        -0.09046773, -0.03252005],
       [-0.00868013, -0.02413091, -0.03615267, ..., -0.15799991,
        -0.09046773, -0.03252005],
       [-0.00868013, -0.02413091, -0.03615267, ..., -0.15799991,
        -0.09046773, -0.03252005]])

In [5]:
x_train_scaled_tensor = torch.from_numpy(x_train_scaled).float().reshape(-1,1,28,28)
x_test_scaled_tensor = torch.from_numpy(x_test_scaled).float().reshape(-1,1,28,28)

y_train_tensor = torch.from_numpy(y_train.astype(np.int64))
y_test_tensor = torch.from_numpy(y_test.astype(np.int64))

In [6]:
train_dataset = TensorDataset(x_train_scaled_tensor,y_train_tensor)

In [7]:
x_train_scaled_tensor.shape

torch.Size([56000, 1, 28, 28])

In [8]:
train_loader = DataLoader(train_dataset,batch_size = 32,shuffle=True)

In [81]:
class BCNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 16, kernel_size=3)
    self.pool = nn.MaxPool2d(2)
    self.conv2 = nn.Conv2d(16, 32, kernel_size=3)
    self.fc1 = nn.Linear(32*5*5, 32)
    self.fc2 = nn.Linear(32, 10)

  def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [82]:
model = BCNet()
print(model._modules)

{'conv1': Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1)), 'pool': MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False), 'conv2': Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1)), 'fc1': Linear(in_features=800, out_features=32, bias=True), 'fc2': Linear(in_features=32, out_features=10, bias=True)}


In [84]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [83]:
print(model)

BCNet(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=800, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=10, bias=True)
)


In [85]:
epochs = 10

for epochs in range(epochs):
  model.train()
  running_loss = 0.0
  for x_batch,y_batch in train_loader:
    optimizer.zero_grad()
    preds = model(x_batch)
    loss = criterion(preds,y_batch)

    loss.backward()

    optimizer.step()
    running_loss += loss.item()
  print(f"Epochs : {epochs + 1} , Loss was {running_loss/len(train_loader)}")

Epochs : 1 , Loss was 0.5121099286803178
Epochs : 2 , Loss was 0.34100028172986846
Epochs : 3 , Loss was 0.29646996776759627
Epochs : 4 , Loss was 0.26709238427558113
Epochs : 5 , Loss was 0.24464135304731982
Epochs : 6 , Loss was 0.22764677597369468
Epochs : 7 , Loss was 0.21237063256278635
Epochs : 8 , Loss was 0.20000656115476576
Epochs : 9 , Loss was 0.18740025514736772
Epochs : 10 , Loss was 0.17888956432363817


In [86]:
with torch.no_grad():
    model.eval()

    preds = model(x_test_scaled_tensor)

    loss = criterion(preds, y_test_tensor).item()

    pred_labels = preds.argmax(dim=1)

    accuracy = (pred_labels == y_test_tensor).float().mean().item()


In [87]:
loss

0.2894870936870575

In [88]:
accuracy


0.9046428799629211